In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
%cd "C:/Users/mateo/Desktop/polymath-jr-drifting"
# This notebook uses DriftXpress; exact drift uses its separate loss/cache API in src.drift.
from src.driftXpress import *
from src.eval import (
    evaluate_latent_model,
    evaluate_generation,
    plot_conditional_mnist_results,
    plot_real_generated_projections,
    plot_loss_history,
    latent_statistics
)
from src.training import (
    compose_objective,
    format_epoch_losses,
    gradient_report,
    per_loss_gradient_norms,
    set_requires_grad,
)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


In [ ]:
transform = transforms.ToTensor()

SELECTED_DIGITS = (0, 8, 2, 4, 7, 9)
DIGIT_TO_CLASS = {digit: index for index, digit in enumerate(SELECTED_DIGITS)}
CLASS_TO_DIGIT = np.asarray(SELECTED_DIGITS)

train_data = datasets.MNIST("data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("data", train=False, download=True, transform=transform)

def keep_selected_digits(dataset):
    selected = torch.tensor(SELECTED_DIGITS, dtype=dataset.targets.dtype)
    mask = torch.isin(dataset.targets, selected)
    dataset.data = dataset.data[mask]
    raw_targets = dataset.targets[mask]
    dataset.targets = torch.tensor(
        [DIGIT_TO_CLASS[int(digit)] for digit in raw_targets],
        dtype=torch.long,
    )
    return dataset

train_data = keep_selected_digits(train_data)
test_data = keep_selected_digits(test_data)

BATCH_SIZE = 500

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Selected digits:", SELECTED_DIGITS)
print("Train samples:", len(train_data))
print("Test samples:", len(test_data))


In [ ]:
class f(nn.Module):
    def __init__(self, input_dim=784, latent_dim=16, num_classes=6, label_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 192), nn.LeakyReLU(0.2),
            nn.Linear(192, 96), nn.LeakyReLU(0.2),
        )
        self.fc_mu = nn.Linear(96, latent_dim)
        self.fc_var = nn.Linear(96, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 96), nn.LeakyReLU(0.2),
            nn.Linear(96, 192), nn.LeakyReLU(0.2),
            nn.Linear(192, input_dim), nn.Sigmoid(),
        )
        self.embedding = nn.Embedding(num_classes, label_dim)
        self.generator = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 48), nn.SiLU(),
            nn.Linear(48, 48), nn.SiLU(),
            nn.Linear(48, 48), nn.SiLU(),
            nn.Linear(48, latent_dim),
        )

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(logvar) * torch.exp(0.5 * logvar)

    def get_latent(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_var(h)
        return mu, logvar, self.reparameterize(mu, logvar)

    def encode_mu(self, x):
        return self.fc_mu(self.encoder(x))

    def decode(self, z):
        return self.decoder(z)

    def generate(self, noise, labels):
        h = torch.cat([noise, self.embedding(labels.long())], dim=1)
        z = self.generator(h)
        return z, self.decode(z)


In [ ]:
model = f(input_dim=784, latent_dim=32, num_classes=6)
print(f"Total parameters: {sum(parameter.numel() for parameter in model.parameters())}")
print(f"Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)}")


In [ ]:
def pretrain_vae(model, data_loader, epochs, lambda_kl, lambda_var, lambda_cov, lr=1e-3):
    """Pretrain the VAE side of one joint model; generator parameters are frozen."""
    vae_parameters = (
        list(model.encoder.parameters())
        + list(model.fc_mu.parameters())
        + list(model.fc_var.parameters())
        + list(model.decoder.parameters())
    )
    generator_parameters = (
        list(model.embedding.parameters()) + list(model.generator.parameters())
    )
    for parameter in generator_parameters:
        parameter.requires_grad_(False)

    optimizer = optim.AdamW(vae_parameters, lr=lr)
    history = {key: [] for key in ("total", "recon", "kl", "var", "covar")}
    print(f"Pretraining the VAE part for {epochs} epochs (generator frozen)...")

    for epoch in range(1, epochs + 1):
        model.train()
        items = {key: 0.0 for key in history}
        for images, _ in data_loader:
            x = images.to(device, non_blocking=True).flatten(start_dim=1)
            mu, logvar, z = model.get_latent(x)
            reconstruction = model.decode(z)
            L_recon = recon_loss(reconstruction, x)
            L_kl = kl_loss(mu, logvar)
            L_var = variance_loss(mu)
            L_cov = covariance_loss(mu)
            loss, _ = compose_objective(
                {"recon": L_recon, "kl": L_kl, "var": L_var, "covar": L_cov},
                {"kl": lambda_kl, "var": lambda_var, "covar": lambda_cov},
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae_parameters, max_norm=5.0)
            optimizer.step()

            items["total"] += loss.item()
            items["recon"] += L_recon.item()
            items["kl"] += L_kl.item()
            items["var"] += L_var.item()
            items["covar"] += L_cov.item()

        for key in items:
            items[key] /= len(data_loader)
            history[key].append(items[key])
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f"-------- VAE pretrain epoch {epoch}/{epochs} --------")
            print(format_epoch_losses(
                {key: items[key] for key in ("recon", "kl", "var", "covar")},
                {"kl": lambda_kl, "var": lambda_var, "covar": lambda_cov},
            ))

    for parameter in generator_parameters:
        parameter.requires_grad_(True)
    return history


def warmup_generator(model, data_loader, epochs, lr=1e-3):
    """Warm up the conditional generator against frozen VAE means."""
    vae_parameters = (
        list(model.encoder.parameters())
        + list(model.fc_mu.parameters())
        + list(model.fc_var.parameters())
        + list(model.decoder.parameters())
    )
    generator_parameters = (
        list(model.embedding.parameters()) + list(model.generator.parameters())
    )
    for parameter in vae_parameters:
        parameter.requires_grad_(False)
    for parameter in generator_parameters:
        parameter.requires_grad_(True)

    optimizer = optim.AdamW(generator_parameters, lr=lr)
    history = {"total": [], "latent_match": []}
    print(f"Warming up the generator for {epochs} epochs (VAE frozen)...")

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        latent_match = 0.0
        for images, class_ids in data_loader:
            x = images.to(device, non_blocking=True).flatten(start_dim=1)
            class_ids = class_ids.to(device, non_blocking=True)
            noise = torch.randn(x.shape[0], model.latent_dim, device=device)
            with torch.no_grad():
                target_mu = model.encode_mu(x)
            generated_latents, _ = model.generate(noise, class_ids)
            loss = F.mse_loss(generated_latents, target_mu)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(generator_parameters, max_norm=5.0)
            optimizer.step()

            total += loss.item()
            latent_match += loss.item()

        total /= len(data_loader)
        latent_match /= len(data_loader)
        history["total"].append(total)
        history["latent_match"].append(latent_match)
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f"-------- Generator warmup epoch {epoch}/{epochs} --------")
            print(f"latent_match={latent_match:.6f}")

    for parameter in vae_parameters:
        parameter.requires_grad_(True)
    return history


def build_conditional_drift_cache(model, data_loader, num_classes, num_landmarks, T):
    """Build the fixed class-conditional DriftXpress cache after pretraining."""
    model.eval()
    references = {class_id: [] for class_id in range(num_classes)}
    with torch.no_grad():
        for images, class_ids in data_loader:
            x = images.to(device, non_blocking=True).flatten(start_dim=1)
            _, _, z = model.get_latent(x)
            class_ids = class_ids.to(device, non_blocking=True)
            for class_id in range(num_classes):
                mask = class_ids == class_id
                if mask.any():
                    references[class_id].append(z[mask].detach())

    cache = {}
    for class_id in range(num_classes):
        if not references[class_id]:
            raise ValueError(f"No reference samples found for class {class_id}.")
        support = torch.cat(references[class_id], dim=0)
        landmarks = select_landmarks(support, num_landmarks)
        K_uu_inv_sqrt = build_nystrom_cache(landmarks, T)
        Ap, bp = pre_compute_summaries(support, landmarks, K_uu_inv_sqrt, T)
        cache[class_id] = (landmarks, K_uu_inv_sqrt, Ap, bp)
    return cache


def conditional_drift_loss(generated, class_ids, cache, T):
    total = generated.new_zeros(())
    field = torch.zeros_like(generated)
    for class_tensor in torch.unique(class_ids):
        class_id = int(class_tensor.item())
        mask = class_ids == class_id
        landmarks, K_uu_inv_sqrt, Ap, bp = cache[class_id]
        class_loss, class_field = drift_loss(
            generated[mask], landmarks, K_uu_inv_sqrt, Ap, bp, T
        )
        total = total + mask.float().mean() * class_loss
        field[mask] = class_field
    return total, field


def training_loop(
    num_epochs,
    T,
    num_landmarks,
    lambda_kl,
    lambda_drift,
    lambda_var,
    lambda_cov,
    lambda_representation=1.0,
    loss_scale=1.0,
    vae_pretrain_epochs=50,
    generator_warmup_epochs=30,
    generator_lr=1e-3,
    lr=1e-4,
):
    global DRIFT_CACHE
    model = f(784, LATENT_DIM, NUM_CLASSES).to(device)
    pretrain_history = pretrain_vae(
        model,
        train_loader,
        vae_pretrain_epochs,
        lambda_kl=lambda_kl,
        lambda_var=lambda_var,
        lambda_cov=lambda_cov,
        lr=lr,
    )

    print("Building the fixed conditional DriftXpress cache once, after pretraining...")
    cache_start = time.perf_counter()
    DRIFT_CACHE = build_conditional_drift_cache(
        model, train_loader, NUM_CLASSES, num_landmarks, T
    )
    print(f"Fixed attraction caches built in {(time.perf_counter() - cache_start) / 60:.2f} minutes")

    generator_warmup_history = warmup_generator(
        model,
        train_loader,
        epochs=generator_warmup_epochs,
        lr=generator_lr,
    )

    optimizer = optim.AdamW(model.parameters(), lr=lr)
    history = {key: [] for key in ("total", "recon", "kl", "drift", "var", "covar", "force")}
    evolution = []
    snapshots = set(range(0, num_epochs + 1, max(1, num_epochs // 5)))
    snapshots.add(num_epochs)
    train_start = time.perf_counter()
    print("Pretrained fully joint conditional training started")

    for epoch in range(num_epochs + 1):
        model.train()
        items = {key: 0.0 for key in history}
        last_pos, last_neg = None, None
        for images, class_ids in train_loader:
            x = images.to(device, non_blocking=True).flatten(start_dim=1)
            class_ids = class_ids.to(device, non_blocking=True)
            noise = torch.randn(x.shape[0], model.latent_dim, device=device)

            mu, logvar, z_pos = model.get_latent(x)
            x_recon = model.decode(z_pos)
            _, x_neg = model.generate(noise, class_ids)
            z_neg = model.encode_mu(x_neg)

            # Exactly Algorithm 3's fully joint route: drift on the re-encoded
            # generated image, so it reaches generator, decoder, and encoder.
            L_drift, V = conditional_drift_loss(z_neg, class_ids, DRIFT_CACHE, T)
            L_recon = recon_loss(x_recon, x)
            L_kl = kl_loss(mu, logvar)
            L_var = variance_loss(mu) + variance_loss(z_neg)
            L_cov = covariance_loss(mu) + covariance_loss(z_neg)
            loss, _ = compose_objective(
                {
                    "recon": L_recon,
                    "kl": L_kl,
                    "drift": L_drift,
                    "var": L_var,
                    "covar": L_cov,
                },
                {
                    "kl": lambda_kl,
                    "drift": lambda_drift,
                    "var": lambda_var,
                    "covar": lambda_cov,
                },
                lambda_representation=lambda_representation,
                loss_scale=loss_scale,
            )
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at epoch {epoch}: {loss.item()}")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            items["total"] += loss.item()
            items["recon"] += L_recon.item()
            items["kl"] += L_kl.item()
            items["drift"] += L_drift.item()
            items["var"] += L_var.item()
            items["covar"] += L_cov.item()
            items["force"] += V.norm(dim=1).mean().item()
            last_pos, last_neg = z_pos.detach(), z_neg.detach()

        for key in items:
            items[key] /= len(train_loader)
            history[key].append(items[key])
        if epoch in snapshots:
            evolution.append({
                "epoch": epoch,
                "pos": last_pos.cpu().numpy(),
                "neg": last_neg.cpu().numpy(),
            })
        if epoch < 20 or epoch % 10 == 0 or epoch == num_epochs:
            print(f"-------- Epoch {epoch} --------")
            print(format_epoch_losses(
                {key: items[key] for key in ("recon", "kl", "drift", "var", "covar")},
                {
                    "kl": lambda_kl,
                    "drift": lambda_drift,
                    "var": lambda_var,
                    "covar": lambda_cov,
                },
                lambda_representation=lambda_representation,
                loss_scale=loss_scale,
            ))
            print("distances", torch.cdist(z_pos[:50], z_pos[:50]).mean().item())
            print(f"average |V|={items['force']:.6f}")
            print(f"latent std positive={last_pos.std(dim=0).mean().item():.6f}")
            print(f"latent std negative={last_neg.std(dim=0).mean().item():.6f}")

    elapsed = (time.perf_counter() - train_start) / 60
    print(f"Finished. Joint training took {elapsed:.2f} minutes")
    return (
        model,
        history["total"],
        history["recon"],
        history["kl"],
        history["drift"],
        history["var"],
        history["covar"],
        history["force"],
        evolution,
        pretrain_history,
        generator_warmup_history,
    )


In [ ]:
# Pretrain exactly the VAE part first, then use Algorithm 3's fully joint
# re-encoded drift route.  The generator is one part of `f`, not a second model.
NUM_CLASSES = 6
LATENT_DIM = 32
VAE_PRETRAIN_EPOCHS = 100
GENERATOR_WARMUP_EPOCHS = 30
NUM_EPOCHS = 200
NUM_LANDMARKS = 200
TEMPERATURE = 2
LAMBDA_KL = 1e-5
LAMBDA_DRIFT = 30.0
LAMBDA_VAR = 0.1
LAMBDA_COV = 0.3
LAMBDA_REPRESENTATION = 1.0
LOSS_SCALE = 1.0  # Keep at 1.0; it is an intentional global gradient-scale test.

(
    f_trained,
    losses,
    recon_losses,
    kl_losses,
    drift_losses,
    var_losses,
    covar_losses,
    average_V,
    evolution,
    pretrain_history,
    generator_warmup_history,
) = training_loop(
    NUM_EPOCHS,
    TEMPERATURE,
    NUM_LANDMARKS,
    LAMBDA_KL,
    LAMBDA_DRIFT,
    LAMBDA_VAR,
    LAMBDA_COV,
    lambda_representation=LAMBDA_REPRESENTATION,
    loss_scale=LOSS_SCALE,
    vae_pretrain_epochs=VAE_PRETRAIN_EPOCHS,
    generator_warmup_epochs=GENERATOR_WARMUP_EPOCHS,
    generator_lr=1e-3,
    lr=1e-4,
)


In [ ]:
# Optional: inspect the actual weighted gradient paths before calling backward.
f_trained.train()
images, class_ids = next(iter(train_loader))
x = images.to(device).flatten(start_dim=1)
class_ids = class_ids.to(device)
noise = torch.randn(x.shape[0], f_trained.latent_dim, device=device)
mu, logvar, z_pos = f_trained.get_latent(x)
x_recon = f_trained.decode(z_pos)
_, x_neg = f_trained.generate(noise, class_ids)
z_neg = f_trained.encode_mu(x_neg)
L_recon = recon_loss(x_recon, x)
L_drift, _ = conditional_drift_loss(z_neg, class_ids, DRIFT_CACHE, TEMPERATURE)
print(per_loss_gradient_norms(
    {"recon": L_recon, "drift": L_drift},
    f_trained,
    {"drift": LAMBDA_DRIFT},
    lambda_representation=LAMBDA_REPRESENTATION,
))


In [ ]:
joint_history = {
    "total": losses,
    "recon": recon_losses,
    "kl": kl_losses,
    "drift": drift_losses,
    "var": var_losses,
    "covar": covar_losses,
    "force": average_V,
}

def plot_training_phases(pretrain_history, warmup_history, joint_history):
    phases = (
        ("VAE pretraining", pretrain_history),
        ("generator warm-up", warmup_history),
        ("fully joint training", joint_history),
    )
    keys = ("total", "recon", "kl", "drift", "var", "covar", "force", "latent_match")
    figure, axes = plt.subplots(4, 2, figsize=(14, 16), sharex=False)
    axes = axes.ravel()
    boundaries = []
    offset = 0
    for phase_name, phase_history in phases:
        phase_length = len(phase_history.get("total", []))
        boundaries.append((offset, offset + phase_length, phase_name))
        offset += phase_length

    for axis, key in zip(axes, keys):
        offset = 0
        plotted = False
        for phase_name, phase_history in phases:
            values = np.asarray(phase_history.get(key, []), dtype=float)
            if values.size:
                x_values = np.arange(offset, offset + values.size)
                axis.plot(x_values, values, label=phase_name)
                plotted = True
            offset += len(phase_history.get("total", []))
        if plotted:
            axis.set_title(key)
            axis.set_yscale("log")
            axis.grid(alpha=0.25)
            axis.legend()
        else:
            axis.axis("off")

    for _, end, _ in boundaries[:-1]:
        for axis in axes:
            axis.axvline(end - 0.5, color="black", linestyle="--", alpha=0.35)
    figure.suptitle("Training losses: pretraining → generator warm-up → joint training")
    figure.tight_layout()
    return figure, axes

fig, axes = plot_training_phases(
    pretrain_history,
    generator_warmup_history,
    joint_history,
)
plt.show()


In [ ]:
import importlib
import src.eval as eval_utils

importlib.reload(eval_utils)

from src.eval import evaluate_latent_model

In [ ]:
# One reusable post-training call: it collects real encoder means and
# conditional generated latents/images, then computes latent SWD/FID and class metrics.
post_eval = evaluate_latent_model(
    f_trained,
    test_loader,
    device=device,
    n_samples=2048,
    n_classes=NUM_CLASSES,
    class_to_label=CLASS_TO_DIGIT,
    include_mmd=False,
    include_fid=True,
)

samples = post_eval["samples"]
real_latents = samples["real_latents"]
generated_latents = samples["generated_latents"]
real_pixels = samples["real_images"]
generated_pixels = samples["generated_images"]
real_class_labels = samples["real_class_ids"]
generated_class_labels = samples["generated_class_ids"]
real_labels = samples["real_labels"]
generated_labels = samples["generated_labels"]
metrics = post_eval["metrics"]
latent_stats = post_eval["statistics"]

print("Latent-space metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.6f}")


In [ ]:
import sys
import inspect

project_root = r"C:\Users\mateo\Desktop\polymath-jr-drifting"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Remove stale src modules without restarting the kernel.
for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

import src.eval as eval_utils

print(eval_utils.__file__)
print(inspect.signature(eval_utils.evaluate_generation))

from src.eval import evaluate_latent_model, evaluate_generation

In [ ]:
# This is the same compact metric interface in pixel space.
# Its FID is a pixel-space Fréchet distance; use evaluate_mnist_generation with a
# frozen evaluator when you specifically want learned MNIST-feature FID.
pixel_metrics = evaluate_generation(
    real_pixels,
    generated_pixels,
    generated_labels=generated_class_labels,
    n_classes=NUM_CLASSES,
    include_mmd=False,
)
print("Pixel-space metrics:")
for name, value in pixel_metrics.items():
    print(f"{name}: {value:.6f}")


In [ ]:
print("Latent statistics:")
for name, value in latent_stats.items():
    print(f"{name}: {value:.6f}")


In [ ]:
fig, axes = plot_conditional_mnist_results(
    f_trained,
    test_loader,
    device=device,
    samples=10,
    use_mean_for_reconstruction=True,
)
plt.show()


In [ ]:
with torch.no_grad():
    generated_images_tensor = torch.as_tensor(
        generated_pixels, dtype=torch.float32, device=device
    )
    generated_reencoded = f_trained.encode_mu(generated_images_tensor).cpu().numpy()

reencoded_stats = latent_statistics(real_latents, generated_reencoded)
print(reencoded_stats)

plot_real_generated_projections(
    real_latents,
    generated_reencoded,
    real_labels=real_labels,
    generated_labels=generated_labels,
    methods=("pca", "umap"),
)
plt.show()

Note that we're using driftXpress so we compute summaries, but as we move the latent space for those landmarks the summaries become obsolete
